# ASL gloss → English : T5-large (Kaggle)

**Sebelum menekan "Run All" / "Save & Run All":**
1. **Accelerator** → GPU (T4 x2 atau P100). Notebook ini hanya memakai 1 GPU.
2. **Internet** → On (Settings → Internet). Dibutuhkan untuk `pip install` dan mengunduh T5-large dari Hugging Face.
3. **Add Input (WAJIB)** → klik *Add Input* → cari dan tambahkan dataset **`unlock-the-power-of-english-asl-with-aslg-pc12-c`** (oleh thedevastator). Tanpa ini, "Save & Run All" gagal karena Kaggle tidak mengizinkan dataset baru di-attach otomatis pada sesi non-interaktif.

Hasil akhir ada di `/kaggle/working` (tab **Output**): model fine-tuned, model ONNX, dan CSV prediksi.
Untuk tes cepat, ubah `MAX_TRAIN_SAMPLES = 2000` di cell konfigurasi.

In [ ]:
import os

# Gunakan 1 GPU saja (hindari DataParallel pada T4 x2). Harus di-set sebelum torch dipakai.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ============================================================
# KONFIGURASI
# ============================================================
MODEL_NAME = "google-t5/t5-large"
MODEL_TAG  = "t5_large"

PREFIX = "translate ASL gloss to English: "
MAX_INPUT_LENGTH  = 128
MAX_TARGET_LENGTH = 128

# Set angka (mis. 2000) untuk tes cepat; None = pakai seluruh data train
MAX_TRAIN_SAMPLES = None

# --- Hyperparameter (disetel untuk t5-large) ---
LEARNING_RATE    = 3e-4
NUM_EPOCHS       = 3
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE  = 8
GRAD_ACCUM_STEPS = 4              # batch efektif = 4 x 4 = 16
OPTIMIZER        = "adafactor"
USE_GRADIENT_CHECKPOINTING = True

# --- Path Kaggle ---
WORKING_DIR = "/kaggle/working"                      # ikut tersimpan di tab Output (maks. 20 GB)
TEMP_ROOT   = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"   # checkpoint sementara

TRAINING_DIR    = f"{TEMP_ROOT}/asl_{MODEL_TAG}_training"
FINAL_MODEL_DIR = f"{WORKING_DIR}/asl_{MODEL_TAG}_finetuned"
ONNX_DIR        = f"{WORKING_DIR}/asl_{MODEL_TAG}_onnx_opencv"
PREDICTIONS_CSV = f"{WORKING_DIR}/asl_{MODEL_TAG}_test_predictions.csv"

DATASET_HANDLE = "thedevastator/unlock-the-power-of-english-asl-with-aslg-pc12-c"
CSV_NAME = "train.csv"

print("Model       :", MODEL_NAME)
print("Model akhir :", FINAL_MODEL_DIR)
print("ONNX        :", ONNX_DIR)

In [ ]:
!pip install -q \
    "transformers==4.46.3" \
    "datasets==3.1.0" \
    "accelerate>=1.1.1" \
    "sentencepiece" \
    "sacrebleu" \
    "pandas" \
    "kagglehub"

In [ ]:
import os
import pandas as pd

def find_file(root, filename):
    for r, _, files in os.walk(root):
        if filename in files:
            return os.path.join(r, filename)
    return None

# 1) Dataset yang di-attach lewat "Add Input"
csv_path = find_file("/kaggle/input", CSV_NAME) if os.path.isdir("/kaggle/input") else None

# 2) Cadangan (hanya berfungsi di sesi interaktif, TIDAK di "Save & Run All")
if csv_path is None:
    try:
        import kagglehub
        dataset_dir = kagglehub.dataset_download(DATASET_HANDLE)
        csv_path = find_file(dataset_dir, CSV_NAME)
    except Exception as e:
        print("kagglehub gagal:", type(e).__name__)

if csv_path is None:
    listing = os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else []
    raise FileNotFoundError(
        f"{CSV_NAME} tidak ditemukan. Isi /kaggle/input: {listing}\n"
        "Klik 'Add Input' dan tambahkan dataset "
        "'unlock-the-power-of-english-asl-with-aslg-pc12-c', lalu jalankan ulang."
    )
print("Memakai file:", csv_path)

df = pd.read_csv(csv_path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

In [ ]:
# Bersihkan teks tanpa mengubah isi kalimat
for col in ["gloss", "text"]:
    df[col] = (
        df[col]
        .astype("string")
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )

# Periksa kembali hasilnya
print("Shape:", df.shape)
print("Kolom:", df.columns.tolist())

display(df.head(10))

In [ ]:
# Sesuaikan nama kolom dengan kode training kita
if "sentence" not in df.columns and "text" in df.columns:
    df = df.rename(columns={"text": "sentence"})

print("Kolom:", df.columns.tolist())
print("Shape:", df.shape)

display(df[["gloss", "sentence"]].head(10))

In [ ]:
from sklearn.model_selection import train_test_split

# Pastikan kolom yang diperlukan tersedia
assert {"gloss", "sentence"}.issubset(df.columns), (
    f"Kolom tidak sesuai: {df.columns.tolist()}"
)

# Bersihkan karakter BOM dan spasi
for col in ["gloss", "sentence"]:
    df[col] = (
        df[col]
        .astype("string")
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )

# Hapus data kosong
df = df.dropna(subset=["gloss", "sentence"])
df = df[
    (df["gloss"] != "") &
    (df["sentence"] != "")
]

# Hapus pasangan gloss-kalimat yang identik
before = len(df)
df = df.drop_duplicates(
    subset=["gloss", "sentence"]
).reset_index(drop=True)

print("Duplikat dihapus:", before - len(df))
print("Total pasangan valid:", len(df))

# Bagi menjadi 80% training, 10% validation, 10% testing
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    shuffle=True,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True,
)

print("\nJumlah data:")
print("Train      :", len(train_df))
print("Validation :", len(val_df))
print("Test       :", len(test_df))

# Periksa beberapa contoh
display(train_df.head(5))

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

# MODEL_NAME dan PREFIX diambil dari cell konfigurasi di atas

# Muat tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Opsional: batasi jumlah data train untuk tes cepat
if MAX_TRAIN_SAMPLES:
    train_df = train_df.sample(
        n=min(MAX_TRAIN_SAMPLES, len(train_df)), random_state=42
    )
    print("Train dibatasi menjadi:", len(train_df))

# Ubah DataFrame Pandas menjadi Hugging Face Dataset
raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(
        train_df[["gloss", "sentence"]].reset_index(drop=True),
        preserve_index=False,
    ),
    "validation": Dataset.from_pandas(
        val_df[["gloss", "sentence"]].reset_index(drop=True),
        preserve_index=False,
    ),
    "test": Dataset.from_pandas(
        test_df[["gloss", "sentence"]].reset_index(drop=True),
        preserve_index=False,
    ),
})

print(raw_datasets)
print("Tokenizer berhasil dimuat:", MODEL_NAME)

In [ ]:
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128

def preprocess_function(examples):
    # Input model: gloss ASL
    inputs = [
        PREFIX + gloss
        for gloss in examples["gloss"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    # Target model: kalimat bahasa Inggris
    targets = tokenizer(
        text_target=examples["sentence"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = targets["input_ids"]

    return model_inputs


tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=["gloss", "sentence"],
    desc="Tokenizing gloss-to-text dataset",
)

print(tokenized_datasets)

In [ ]:
sample = tokenized_datasets["train"][0]

print("Input IDs:")
print(sample["input_ids"])

print("\nDecoded input:")
print(
    tokenizer.decode(
        sample["input_ids"],
        skip_special_tokens=True,
    )
)

print("\nDecoded target:")
print(
    tokenizer.decode(
        sample["labels"],
        skip_special_tokens=True,
    )
)

print("\nJumlah data:")
for split in tokenized_datasets:
    print(split, ":", len(tokenized_datasets[split]))

In [ ]:
import os
import inspect
import torch

from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

# Pastikan notebook Kaggle menggunakan GPU
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU belum aktif. Buka Settings > Accelerator dan pilih GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))

# T5 tidak stabil di fp16, jadi pakai bf16 jika didukung, selain itu fp32
# bf16 asli hanya ada di GPU Ampere ke atas (compute capability >= 8). T4/P100 tidak punya.
USE_BF16 = torch.cuda.get_device_capability(0)[0] >= 8
if not USE_BF16:
    print(
        "PERINGATAN: GPU ini tidak mendukung bf16 (mis. T4). "
        "Training berjalan di fp32 (lebih lambat). "
        "Ini normal di Kaggle (T4/P100)."
    )

# Muat model pretrained
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Padding dilakukan secara dinamis dalam setiap batch
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    pad_to_multiple_of=8,
)

# TRAINING_DIR diambil dari cell konfigurasi di atas

args_kwargs = dict(
    output_dir=TRAINING_DIR,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    optim=OPTIMIZER,
    warmup_ratio=0.03,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_steps=100,
    bf16=USE_BF16,
    fp16=False,  # fp16 tidak stabil pada T5 (loss bisa NaN)
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    seed=42,
    dataloader_num_workers=2,
)

# Menyesuaikan nama argumen dengan versi Transformers
training_params = inspect.signature(
    Seq2SeqTrainingArguments.__init__
).parameters

if "eval_strategy" in training_params:
    args_kwargs["eval_strategy"] = "epoch"
else:
    args_kwargs["evaluation_strategy"] = "epoch"

training_args = Seq2SeqTrainingArguments(**args_kwargs)

# Menyesuaikan API Trainer dengan versi Transformers
trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

trainer_params = inspect.signature(Seq2SeqTrainer.__init__).parameters

if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

print("Model siap dilatih:", MODEL_NAME)
print("Output training :", TRAINING_DIR)
print(
    "Batch efektif    :",
    TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS,
)
print("Train:", len(tokenized_datasets["train"]))
print("Validation:", len(tokenized_datasets["validation"]))

In [ ]:
train_result = trainer.train()

print("Training selesai.")
print("Training loss:", train_result.training_loss)

In [ ]:
# trainer sudah memuat checkpoint terbaik berdasarkan validation loss
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("Model tersimpan di:", FINAL_MODEL_DIR)
print("Isi direktori:", os.listdir(FINAL_MODEL_DIR))

In [ ]:
import torch
import pandas as pd

from tqdm.auto import tqdm
from sacrebleu.metrics import BLEU, CHRF

model = trainer.model
model.config.use_cache = True  # dimatikan oleh gradient checkpointing saat training
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

test_data = raw_datasets["test"]

predictions = []
references = []
test_glosses = []

BATCH_SIZE = EVAL_BATCH_SIZE

for start in tqdm(
    range(0, len(test_data), BATCH_SIZE),
    desc="Evaluating test set",
):
    end = min(start + BATCH_SIZE, len(test_data))
    batch = test_data[start:end]

    prompts = [
        PREFIX + gloss
        for gloss in batch["gloss"]
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    ).to(device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=2,
            do_sample=False,
        )

    batch_predictions = tokenizer.batch_decode(
        output_ids,
        skip_special_tokens=True,
    )

    predictions.extend(batch_predictions)
    references.extend(batch["sentence"])
    test_glosses.extend(batch["gloss"])

# Hitung metrik
bleu_metric = BLEU(lowercase=True)
chrf_metric = CHRF(word_order=2, lowercase=True)

bleu_score = bleu_metric.corpus_score(
    predictions,
    [references],
)

chrf_score = chrf_metric.corpus_score(
    predictions,
    [references],
)

print(f"Test BLEU: {bleu_score.score:.2f}")
print(f"Test chrF++: {chrf_score.score:.2f}")

# Periksa hasil terjemahan secara manual
results_df = pd.DataFrame({
    "gloss": test_glosses,
    "reference": references,
    "prediction": predictions,
})

display(results_df.head(20))

# Simpan seluruh prediksi agar bisa dianalisis
results_df.to_csv(
    PREDICTIONS_CSV,
    index=False,
)
print("Prediksi disimpan di:", PREDICTIONS_CSV)

In [ ]:
!pip install -q "optimum-onnx[onnxruntime]" onnx

In [ ]:
%pip uninstall -y diffusers

In [ ]:
import gc, shutil
import torch

# Bebaskan memori GPU/RAM sebelum ekspor
for name in ("trainer", "model"):
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()

assert os.path.isdir(FINAL_MODEL_DIR), f"Model tidak ditemukan: {FINAL_MODEL_DIR}"
shutil.rmtree(ONNX_DIR, ignore_errors=True)

!optimum-cli export onnx \
    --model "{FINAL_MODEL_DIR}" \
    --task text2text-generation \
    --opset 17 \
    "{ONNX_DIR}"

In [ ]:
from pathlib import Path
import onnx

ONNX_PATH = Path(ONNX_DIR)

for path in sorted(ONNX_PATH.rglob("*")):
    if path.is_file():
        print(f"{path.name:35s} {path.stat().st_size / 1e6:10.2f} MB")

assert (ONNX_PATH / "encoder_model.onnx").exists()
assert (ONNX_PATH / "decoder_model.onnx").exists()

# Validasi kedua graph (pakai path agar model > 2 GB dengan external data tetap terbaca)
for filename in ["encoder_model.onnx", "decoder_model.onnx"]:
    onnx.checker.check_model(str(ONNX_PATH / filename))
    print(f"[VALID] {filename}")

In [ ]:
import torch
from transformers import AutoTokenizer
from optimum.onnxruntime import ORTModelForSeq2SeqLM

onnx_tokenizer = AutoTokenizer.from_pretrained(ONNX_DIR)

onnx_model = ORTModelForSeq2SeqLM.from_pretrained(
    ONNX_DIR,
    provider="CPUExecutionProvider",
    use_cache=False,
    use_merged=False,
)
print("ONNX model loaded successfully.")

for i in range(min(5, len(raw_datasets["test"]))):
    sample = raw_datasets["test"][i]

    inputs = onnx_tokenizer(
        PREFIX + sample["gloss"],
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    )

    with torch.inference_mode():
        output_ids = onnx_model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=2,
            do_sample=False,
            use_cache=False,
        )

    prediction = onnx_tokenizer.decode(output_ids[0], skip_special_tokens=True)

    print(f"\nContoh {i + 1}")
    print("Gloss   :", sample["gloss"])
    print("Target  :", sample["sentence"])
    print("Prediksi:", prediction)

In [ ]:
import os

print("Isi /kaggle/working (akan muncul di tab Output):\n")
for root, _, files in os.walk(WORKING_DIR):
    for name in sorted(files):
        p = os.path.join(root, name)
        print(f"{os.path.relpath(p, WORKING_DIR):60s} {os.path.getsize(p) / 1e6:10.1f} MB")